# Estudo de ablação — dual task (CN×AD, sMCI×pMCI)

Hipocampo L/R × T1/T2/T3 (features absolutas, sem deltas).

| Task | Classes |
|------|---------|
| `cn_ad` | CN (0) vs AD (1) |
| `smci_pmci` | sMCI (0) vs pMCI (1) |

**Modalidades** (`csvs/longitudinal_4_groups/ablation/{roi}/`):

| ID | CSV long | Conteúdo |
|----|----------|----------|
| `vol` | `vol_long.csv` | volumétrico |
| `texture` | `rad_long.csv` | textura |
| `disp` | `disp_long.csv` | deslocamento |
| `all` | `merge_long.csv` | merge filtrado |

**Fluxo por fold:** CSV long → split `ID_PT` → ComBat opcional → pivot wide → CV 5×5.

Lógica em `ablation_runner.py`. Ordem: config → overview → execução → análise.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from ablation_prep import ROI_FILTER_DEFAULT
from ablation_runner import MODALITIES, SELECTION_MODES, TASKS, run_full_ablation_suite

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
ROI = ROI_FILTER_DEFAULT
BASE_DIR = Path(f"csvs/longitudinal_4_groups/ablation/{ROI}")
RESULTS_DIR = Path("csvs/longitudinal_4_groups/ablation_results")
RESULTS_PATH = RESULTS_DIR / "ablation_results_all.csv"
SUMMARY_PATH = RESULTS_DIR / "ablation_summary.csv"

# ComBat por fold — (False,) só sem; (False, True) compara ambos
WITH_COMBAT_FLAGS = (False, True)

RUN_TASKS = ("cn_ad", "smci_pmci")
RUN_MODALITIES = tuple(MODALITIES.keys())
RUN_SELECTION_MODES = ("raw",)  # "raw", "filters", "mrmr"
MODELS = ("svm", "rf", "xgb", "mlp")

METRIC_COLS = [
    "accuracy", "auc", "auc_pr", "bal_acc", "mcc",
    "sens_pos", "spec_neg", "f1_pos",
]

In [ ]:
# Visão geral dos CSVs long de entrada (por task)
for mod_id in RUN_MODALITIES:
    cfg = MODALITIES[mod_id]
    long_path = BASE_DIR / cfg["long"]
    if not long_path.is_file():
        print(f"[{mod_id}] AUSENTE: {long_path}")
        continue
    df = pd.read_csv(long_path)
    print(f"\n{mod_id.upper():8s} — {cfg['label']} — {long_path.name}")
    print(f"  linhas={len(df):,} | imagens={df['ID_IMG'].nunique():,} | pacientes={df['ID_PT'].nunique():,}")
    for task_id in RUN_TASKS:
        task = TASKS[task_id]
        sub = df[df["GROUP"].astype(str).isin(task.groups)]
        n_pt = sub["ID_PT"].nunique()
        counts = sub.groupby("ID_PT")["GROUP"].first().value_counts().to_dict()
        print(f"  {task_id}: pacientes={n_pt} | {counts}")

In [ ]:
# Execução: task × modalidade × ComBat × seleção × modelo
df_ablation = run_full_ablation_suite(
    base_dir=BASE_DIR,
    roi=ROI,
    tasks=RUN_TASKS,
    modalities=RUN_MODALITIES,
    models=MODELS,
    selection_modes=RUN_SELECTION_MODES,
    with_combat_flags=WITH_COMBAT_FLAGS,
    results_dir=RESULTS_DIR,
    seed=SEED,
)
print(f"\nResultados: {RESULTS_PATH}")
df_ablation.head()

In [ ]:
# Resumo agregado por task, modalidade, ComBat e modelo
group_cols = ["task", "modality", "modality_label", "with_combat", "model_key", "best_model"]
if "selection_mode" in df_ablation.columns:
    group_cols = ["selection_mode"] + group_cols

summary = (
    df_ablation
    .groupby(group_cols, as_index=False)
    .agg(
        n_folds=("fold", "count"),
        n_features_mean=("n_features_selected", "mean"),
        **{
            f"{col}_{stat}": (col, stat)
            for col in METRIC_COLS
            for stat in ("mean", "std")
        },
    )
)
sort_cols = ["auc_mean", "task", "modality", "model_key"]
if "selection_mode" in summary.columns:
    sort_cols = ["selection_mode"] + sort_cols
summary = summary.sort_values(sort_cols, ascending=[False] * len(sort_cols))
summary.to_csv(SUMMARY_PATH, index=False)
print(f"Resumo salvo em: {SUMMARY_PATH}\n")

display_cols = [
    "task", "modality", "with_combat", "model_key", "best_model",
    "n_features_mean",
    "auc_mean", "auc_std",
    "mcc_mean", "mcc_std",
    "bal_acc_mean", "bal_acc_std",
    "sens_pos_mean", "sens_pos_std",
    "spec_neg_mean", "spec_neg_std",
    "f1_pos_mean", "f1_pos_std",
]
if "selection_mode" in summary.columns:
    display_cols = ["selection_mode"] + display_cols
display(summary[display_cols])

In [ ]:
# Melhor configuração por task × modalidade (maior AUC médio)
best_group = ["task", "modality", "with_combat"]
if "selection_mode" in summary.columns:
    best_group = ["selection_mode"] + best_group

best_per_modality = (
    summary
    .sort_values("auc_mean", ascending=False)
    .groupby(best_group, as_index=False)
    .first()
)
print("Melhor modelo por task × modalidade (AUC médio nos 5 folds externos):\n")
for _, row in best_per_modality.iterrows():
    mode_prefix = f"[{row['selection_mode']}] " if "selection_mode" in row.index else ""
    combat_tag = "combat" if row.get("with_combat") else "nocombat"
    print(
        f"  {mode_prefix}{row['task']:10s} {row['modality']:8s} ({combat_tag}) "
        f"→ {row['model_key'].upper():4s} / {row['best_model']:22s} | "
        f"AUC={row['auc_mean']:.3f}±{row['auc_std']:.3f} | "
        f"MCC={row['mcc_mean']:.3f}±{row['mcc_std']:.3f} | "
        f"sens={row['sens_pos_mean']:.3f}±{row['sens_pos_std']:.3f} | "
        f"spec={row['spec_neg_mean']:.3f}±{row['spec_neg_std']:.3f} | "
        f"F1={row['f1_pos_mean']:.3f}±{row['f1_pos_std']:.3f} | "
        f"feat≈{row['n_features_mean']:.0f}"
    )

In [ ]:
# Comparação visual: AUC por task, modalidade e ComBat
mod_order = ["vol", "texture", "disp", "all"]

for task_id in df_ablation["task"].unique():
    sub_task = df_ablation[df_ablation["task"] == task_id]
    for with_combat in sorted(sub_task["with_combat"].unique()):
        sub = sub_task[sub_task["with_combat"] == with_combat]
        pivot_auc = sub.pivot_table(
            index="model_key", columns="modality", values="auc", aggfunc="mean"
        ).reindex(columns=mod_order)

        fig, ax = plt.subplots(figsize=(8, 4))
        pivot_auc.plot(kind="bar", ax=ax, rot=0)
        combat_tag = "ComBat" if with_combat else "sem ComBat"
        ax.set_title(f"{task_id} — {combat_tag} — AUC médio (5 folds)")
        ax.set_ylabel("AUC")
        ax.legend(title="Modalidade")
        ax.axhline(0.5, color="gray", ls="--", lw=0.8)
        plt.tight_layout()
        out = RESULTS_DIR / f"ablation_auc_{task_id}_{'combat' if with_combat else 'nocombat'}.png"
        plt.savefig(out, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Figura: {out}")

In [ ]:
# Estabilidade das features selecionadas (frequência por task × modalidade)
from collections import Counter


def feature_stability(df_mod: pd.DataFrame, top_n: int = 15) -> pd.DataFrame:
    counter: Counter = Counter()
    for feats_json in df_mod["selected_features"]:
        counter.update(json.loads(feats_json))
    total = len(df_mod)
    return pd.DataFrame([
        {"feature": feat, "count": cnt, "freq": cnt / total}
        for feat, cnt in counter.most_common(top_n)
    ])


for task_id in df_ablation["task"].unique():
    for mod_id in RUN_MODALITIES:
        df_mod = df_ablation[
            (df_ablation["task"] == task_id) & (df_ablation["modality"] == mod_id)
        ]
        if df_mod.empty:
            continue
        stab = feature_stability(df_mod)
        label = MODALITIES[mod_id]["label"]
        print(f"\nTop features — {task_id} / {mod_id.upper()} ({label}):")
        display(stab)

## Carregar resultados salvos

Se a execução acima já foi feita, use a célula abaixo para reanalisar sem re-treinar.

In [ ]:
df_ablation = pd.read_csv(RESULTS_PATH)
summary = pd.read_csv(SUMMARY_PATH)
df_ablation.shape, summary.shape